## Add tags for to the csv, categorize by breakfast, lunch, dinner and allergies/ dietary restriction categories

In [1]:
import pandas as pd 
import json 

In [12]:
def infer_allergens(food_name, category):
    text = f"{food_name} {category}".lower()
    allergens = set()

    if any(word in text for word in ["milk", "cheese", "yogurt", "butter", "cream", "dairy"]):
        allergens.add("dairy")

    if any(word in text for word in ["wheat", "bread", "toast", "pasta", "bagel", "cereal", "flour", "pancake", "pizza", "bun", "noodle"]):
        allergens.add("gluten")

    if any(word in text for word in ["nut", "almond", "peanut", "cashew", "walnut"]):
        allergens.add("nuts")

    if any(word in text for word in ["fish", "seafood", "salmon", "tuna", "shrimp", "crab", "lobster"]):
        allergens.add("seafood")

    if any(word in text for word in ["egg", "eggs", "omelet", "scrambled"]):
        allergens.add("egg")

    if any(word in text for word in ["soy", "tofu", "edamame", "tempeh"]):
        allergens.add("soy")

    return sorted(allergens)

In [14]:
# robust CSV load to avoid tokenizing errors
df_path = "src/data/daily_food_nutrition_dataset.csv"
df = pd.read_csv(df_path, engine="python", on_bad_lines="skip")

# coerce expected numeric columns to numbers (invalid parses become NaN)
_numeric_cols = [
    "Calories (kcal)", "Protein (g)", "Carbohydrates (g)", "Fat (g)",
    "Fiber (g)", "Sugars (g)", "Sodium (mg)", "Cholesterol (mg)"
]
for c in _numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
cleaned_foods = []

for index, row in df.iterrows():
    food_name = row["Food_Item"]
    category = row["Category"]

    allergens = infer_allergens(food_name, category)

    cleaned_foods.append({
        "id": f"food-{index + 1}",
        "name": food_name,
        "originalCategory": category,
        "allergens": allergens,
        "nutrition": {
            "calories": row["Calories (kcal)"],
            "protein": row["Protein (g)"],
            "carbs": row["Carbohydrates (g)"],
            "fat": row["Fat (g)"],
            "fiber": row["Fiber (g)"],
            "sugar": row["Sugars (g)"],
            "sodium": row["Sodium (mg)"],
            "cholesterol": row["Cholesterol (mg)"]
        },
        "mealTypes": [row["Meal_Type"].lower()]
    })

with open("src/data/foods.json", "w") as f:
    json.dump(cleaned_foods, f, indent=2)